In [ ]:
print("Hola desde Jupyter")

Hola desde Jupyter


In [1]:
import os
import mysql.connector
import pandas as pd
from dotenv import load_dotenv

# 1. Cargamos las variables de entorno de forma segura
load_dotenv()

try:
    print(f"Intentando conectar a la base de datos de forma segura...")
    
    # 2. Conectamos usando os.getenv() para no mostrar contraseñas en el código
    conexion = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        port=int(os.getenv("DB_PORT")),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME"),
        connection_timeout=5
    )
    print("¡CONECTADO EXITOSAMENTE REGLAMENTARIO! 🎉\n")
    
    # 3. Validamos las tablas existentes
    cursor = conexion.cursor()
    cursor.execute("SHOW TABLES;")
    tablas = cursor.fetchall()
    
    print(f"Tablas sincronizadas desde '{os.getenv('DB_NAME')}':")
    for t in tablas:
        print(f"  • {t[0]}")
        
    conexion.close()

except mysql.connector.Error as error:
    print(f"\nError de conexión: {error}")
    print("Chequeá que el archivo .env esté bien guardado y en la misma carpeta.")

Intentando conectar a la base de datos de forma segura...
¡CONECTADO EXITOSAMENTE REGLAMENTARIO! 🎉

Tablas sincronizadas desde 'final':
  • autor
  • ejemplar
  • genero
  • libro
  • libro_autor
  • libro_genero
  • prestamo
  • sancion
  • socio
  • tipo_sancion


In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

# Carga las variables del archivo .env que me mostraste en la captura
load_dotenv()

# Lee la clave de forma segura sin que quede escrita a mano
API_KEY_GROQ = os.getenv("API_KEY_GROQ")

client = Groq(api_key=API_KEY_GROQ)

PROMPT_SISTEMA = """
Eres un asistente experto en bases de datos relacionales de MySQL. 
Tu única tarea es traducir preguntas en lenguaje natural (español) a consultas SQL válidas basadas exclusivamente en este esquema:

Tablas del sistema:
- autor (id_autor, nombre, apellido, nacionalidad)
- genero (id_genero, nombre, descripcion)
- libro (isbn, titulo, anio_publicacion, stock_total, stock_disponible)
- libro_autor (isbn, id_autor)
- libro_genero (isbn, id_genero)
- ejemplar (id_ejemplar, isbn, nro_ejemplar, estado_fisico)
- socio (id_socio, dni, nombre, apellido, email, fecha_alta, estado)
- prestamo (id_prestamo, id_socio, id_ejemplar, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado)
- sancion (id_sancion, id_socio, tipo, fecha_inicio, fecha_fin, motivo)

Reglas de Negocio Cruciales:
1. Para saber si un préstamo está vencido, debes consultar la tabla `prestamo` filtrando por `estado = 'VENCIDO'`. NO uses la tabla sancion para esto a menos que se pida explícitamente.
2. Si se piden datos de los socios, une `socio` con `prestamo` usando `id_socio`.
3. Los únicos valores posibles para la columna `estado` en la tabla `socio` son 'ACTIVO' o 'SUSPENDIDO'. Si te piden socios suspendidos, usa exactamente 'SUSPENDIDO'.
4. Responde ÚNICAMENTE con la consulta SQL pura. NO uses bloques de código Markdown (```sql), ni agregues texto extra. Solo el texto del SELECT.
"""

def consultar_ia(pregunta_usuario):
    # Usamos el modelo recomendado en la guía técnica del TP
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": pregunta_usuario}
        ],
        temperature=0.0
    )
    return completion.choices[0].message.content.strip()

# --- PRUEBA DEL AGENTE ---
pregunta = "¿Cuáles son los títulos de los libros que tenemos en la biblioteca?"
try:
    sql_generado = consultar_ia(pregunta)
    print("🤖 Pregunta efectuada:", pregunta)
    print("💻 SQL generado por el Agente:")
    print(sql_generado)
except Exception as e:
    print(f"Ocurrió un error con la IA: {e}")

🤖 Pregunta efectuada: ¿Cuáles son los títulos de los libros que tenemos en la biblioteca?
💻 SQL generado por el Agente:
SELECT titulo FROM libro


In [7]:
import os
import mysql.connector
import pandas as pd

def agente_responder(pregunta_usuario):
    print(f"❓ Pregunta del usuario: {pregunta_usuario}")
    
    try:
        # 1. Le pedimos el SQL a la IA
        sql = consultar_ia(pregunta_usuario)
        print(f"💻 SQL Generado: {sql}\n")
        
        # 2. Conectamos a MySQL
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            port=int(os.getenv("DB_PORT")),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        
        # 3. Ejecutamos y pasamos a DataFrame
        df = pd.read_sql(sql, conexion)
        conexion.close()
        
        if not df.empty:
            print("📊 Resultados obtenidos de la base de datos:")
            return df
        else:
            print("⚠ La consulta se ejecutó bien, pero no devolvió ningún registro.")
            return None
            
    except Exception as error:
        print(f"❌ Error al procesar la solicitud: {error}")
        return None

# --- PRUEBA DE FUEGO INTEGRADORA ---
agente_responder("¿Cuáles son los títulos de los libros que tenemos en la biblioteca?")

❓ Pregunta del usuario: ¿Cuáles son los títulos de los libros que tenemos en la biblioteca?
💻 SQL Generado: SELECT titulo FROM libro

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,Y no quedó ninguno
1,Asesinato en el Orient Express
2,Sapiens: De animales a dioses
3,El Señor de los Anillos: La Comunidad del Anillo
4,El Señor de los Anillos: Las Dos Torres
5,El Señor de los Anillos: El Retorno del Rey
6,Cien años de soledad
7,El Resplandor
8,Ficciones
9,Cosmos


In [8]:
agente_responder("Mostrame los títulos de los libros escritos por el autor Isaac Asimov")

❓ Pregunta del usuario: Mostrame los títulos de los libros escritos por el autor Isaac Asimov
💻 SQL Generado: SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_autor la ON l.isbn = la.isbn 
JOIN autor a ON la.id_autor = a.id_autor 
WHERE a.nombre = 'Isaac' AND a.apellido = 'Asimov'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,Fundación
1,Fundación e Imperio


In [12]:
agente_responder("¿Qué socios tienen préstamos vencidos actualmente? Mostrame sus nombres y apellidos")

❓ Pregunta del usuario: ¿Qué socios tienen préstamos vencidos actualmente? Mostrame sus nombres y apellidos
💻 SQL Generado: SELECT DISTINCT s.nombre, s.apellido 
FROM socio s 
JOIN prestamo p ON s.id_socio = p.id_socio 
WHERE p.estado = 'VENCIDO'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,nombre,apellido
0,Juan,Rodríguez
1,Valentina,Álvarez
2,Mateo,Romero
3,Emma,Sánchez


In [15]:
# ====================================================================
# DEMOSTRACIÓN FINAL DEL AGENTE (PRUEBAS OBLIGATORIAS DEL TP)
# ====================================================================

print("--- PRUEBA 1: Búsqueda con Joins y agregación ---")
display(agente_responder("¿Qué libros escribió el autor J.R.R. Tolkien?"))
print("-" * 60)

print("\n--- PRUEBA 2: Filtros de estados y fechas ---")
display(agente_responder("Mostrame los socios que están suspendidos"))
print("-" * 60)

print("\n--- PRUEBA 3: Funciones de agregación y ordenamiento ---")
display(agente_responder("¿Cuáles son los 3 libros con mayor stock total en la biblioteca?"))
print("-" * 60)

print("\n--- PRUEBA 4: Cruce complejo de tres tablas ---")
display(agente_responder("Listá los títulos de los libros que pertenecen al género Fantasía"))
print("-" * 60)

--- PRUEBA 1: Búsqueda con Joins y agregación ---
❓ Pregunta del usuario: ¿Qué libros escribió el autor J.R.R. Tolkien?
💻 SQL Generado: SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_autor la ON l.isbn = la.isbn 
JOIN autor a ON la.id_autor = a.id_autor 
WHERE a.nombre = 'J.R.R.' AND a.apellido = 'Tolkien'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,El Señor de los Anillos: La Comunidad del Anillo
1,El Señor de los Anillos: Las Dos Torres
2,El Señor de los Anillos: El Retorno del Rey


------------------------------------------------------------

--- PRUEBA 2: Filtros de estados y fechas ---
❓ Pregunta del usuario: Mostrame los socios que están suspendidos
💻 SQL Generado: SELECT * FROM socio WHERE estado = 'SUSPENDIDO'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


...

,titulo,stock_total
0,1984,5
1,El Señor de los Anillos: La Comunidad del Anillo,5
2,Sapiens: De animales a dioses,4


------------------------------------------------------------

--- PRUEBA 4: Cruce complejo de tres tablas ---
❓ Pregunta del usuario: Listá los títulos de los libros que pertenecen al género Fantasía
💻 SQL Generado: SELECT DISTINCT l.titulo 
FROM libro l 
JOIN libro_genero lg ON l.isbn = lg.isbn 
JOIN genero g ON lg.id_genero = g.id_genero 
WHERE g.nombre = 'Fantasía'

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,titulo
0,El Señor de los Anillos: La Comunidad del Anillo
1,El Señor de los Anillos: Las Dos Torres
2,El Señor de los Anillos: El Retorno del Rey
3,Juego de Tronos
4,Choque de Reyes


------------------------------------------------------------


In [10]:
agente_responder("¿Cuántos libros hay de cada género en la biblioteca?")

❓ Pregunta del usuario: ¿Cuántos libros hay de cada género en la biblioteca?
💻 SQL Generado: SELECT g.nombre, COUNT(l.isbn) FROM libro l JOIN libro_genero lg ON l.isbn = lg.isbn JOIN genero g ON lg.id_genero = g.id_genero GROUP BY g.nombre

📊 Resultados obtenidos de la base de datos:


C:\Users\Lucas\AppData\Local\Temp\ipykernel_8868\2395795252.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conexion)


,nombre,COUNT(l.isbn)
0,Ciencia Ficción,5
1,Divulgación Científica,4
2,Fantasía,5
3,Novela Histórica,3
4,Terror,2
